In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import time
import datetime
import random
import cv2
import warnings
import math
from tabulate import tabulate
import scipy.io as sio
sys.path.append('../')

In [ ]:
from MLVcode.load_mat import load_mat
from MLVcode.drawLinedrawing import drawLinedrawing

In [ ]:
filename = '../dataSets/TorontoScenes/cities_vecLD.mat'
# Given the above keys, we access 77th row of the data
data_keys = {'originalImage','imsize', 'lineMethod', 'numContours',                
            'contours', 'orientations', 'lengths',   'contourLengths',             
            'curvatures', 'junctions', 'parallelism',  'parallelismMeans',           
            'parallelism_allX', 'parallelism_allY','parallelism_allScores', 
            'parallelismBins', 'parallelismHistograms', 'parallelismNormHistograms',  
            'parallelismSumHistogram', 'parallelismNormSumHistogram',
            'separation', 'separationMeans', 'separation_allX',  'separation_allY',            
            'separation_allScores', 'separationBins',             
            'separationHistograms', 'separationNormHistograms',   
            'separationSumHistogram', 'separationNormSumHistogram', 
            'mirror', 'mirrorMeans', 'mirror_allX', 'mirror_allY',                
            'mirror_allScores', 'mirrorBins', 'mirrorHistograms', 
            'mirrorNormHistograms', 'mirrorSumHistogram', 'mirrorNormSumHistogram'}

In [ ]:
data = load_mat(filename)
print(data['vecLD']['contourLengths'][0][0][76])
print(data['vecLD']['contours'][0][8].shape)
print(data['vecLD']['contourLengths'][0][1].shape)

In [ ]:
cute = {}
# Assign the data to the dictionary and define them with empty value
for key in data_keys:
    cute[key] = data['vecLD'][key][0][76]
print(cute['contours'][0].shape)

In [ ]:
drawLinedrawing(cute)

In [ ]:
from MLVcode.computeOrientation import computeOrientation
from MLVcode.computeLength import computeLength
from MLVcode.computeCurvature import computeCurvature
from MLVcode.computeJunctions import computeJunctions
from MLVcode.computeContourProperties import computeContourProperties

### Draw functions

In [ ]:
from MLVcode.drawJunctions import drawJunctions
from MLVcode.drawLinedrawingProperty import drawLinedrawingProperty
from MLVcode.computeColorIndex import computeColorIndex
from MLVcode.drawMATproperty import drawMATproperty
from MLVcode.drawThisProperty import drawThisProperty
from MLVcode.drawAllProperties import drawAllProperties

### Testing

In [ ]:
# cute = computeOrientation(cute)
# cute = computeLength(cute)
temp_ld = computeContourProperties(cute)

In [ ]:
drawAllProperties(temp_ld, 'subplot', ['parallelism', 'separation', 'mirror'])

In [ ]:
drawAllProperties(temp_ld, 'subplot', ['original','orientation','length','junctions',
                                    'mirror','parallelism','separation'])

# Stats calculation

In [ ]:
# Get a copy of cute 
temp_cute = cute.copy()

In [ ]:
from MLVcode.getOrientationStats import getOrientationStats
from MLVcode.getLengthStats import getLengthStats
from MLVcode.getCurvatureStats import getCurvatureStats
from MLVcode.getHorizontalVerticalStats import getHorizontalVerticalStats
from MLVcode.getJunctionStats import getJunctionStats
from MLVcode.getContourPropertiesStats import getContourPropertiesStats

In [ ]:
temp_cute,histograms,bins,statsNames = getContourPropertiesStats(temp_cute, whichStats=['curvature','length','orientation','junctions'])

In [ ]:
angles = []
for i in range(len(temp_cute['junctions'])):
    angles.append(temp_cute['junctions'][i]['minAngle'])
angleHist, _ = np.histogram(angles, bins=8)
# Plot the histogram
plt.figure()
plt.bar(np.arange(8), angleHist)
plt.xlabel('Angle')
plt.ylabel('Frequency')
plt.title('Histogram of Angles')
plt.show()

In [ ]:
# Plot histogram of cute['normSumCurvatureHistogram'] using plot
plt.figure(figsize=(8,6))
plt.bar(range(len(temp_cute['normSumCurvatureHistogram'])), temp_cute['normSumCurvatureHistogram'])
plt.title('Curvature Histogram')
plt.xlabel('Curvature Bins')
plt.ylabel('Contour Pixels')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(range(len(temp_cute['normSumOrientationHistogram'])), temp_cute['normSumOrientationHistogram'])
plt.title('Orientation Histogram')
plt.xlabel('Orientation Bins')
plt.ylabel('Contour Pixels')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(range(len(temp_cute['normSumLengthHistogram'])), temp_cute['normSumLengthHistogram'])
plt.title('Length Histogram')
plt.xlabel('Length Bins')
plt.ylabel('Contour Pixels')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(range(len(temp_cute['normJunctionTypeHistogram'])), temp_cute['normJunctionTypeHistogram'])
plt.title('Junction Histogram')
plt.xlabel('Junction Type')
plt.xticks(range(len(temp_cute['junctionTypeBins'])), temp_cute['junctionTypeBins'])
plt.ylabel('Per 10000 Contour Pixels')
plt.show()

## Split functions

In [ ]:
from MLVcode.splitLDbyProperties import splitLDbyProperties

In [ ]:
# Split image into two versions based on selected properties
# Split image by curvature and orientation (50% proportion of pixels - default)
top,bottom = splitLDbyProperties(temp_cute,['length'])
drawLinedrawing(top)
drawLinedrawing(bottom)

In [ ]:
top, bottom = splitLDbyProperties(temp_cute, ['length'], 0.25)
print("Top:")
drawLinedrawing(top)
print("Bottom:")
drawLinedrawing(bottom)

In [ ]:
top, bottom = splitLDbyProperties(cute, ['orientation'],fraction=0.550)
drawLinedrawing(top)
drawLinedrawing(bottom)

In [ ]:
# THere is some error when fraction=0.15. Need to check that
top, bottom = splitLDbyProperties(temp_cute, ['curvature'],fraction=0.30)
drawLinedrawing(top)
drawLinedrawing(bottom)